# Designing an A/B test

Core stage, lesson C14. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** write the plan of an A/B test before the data, size a test (baseline, smallest lift worth catching, people, weeks), check the sample ratio before reading any result, find out whether a lift lasts, and read the result against the rule that was written first.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers

import numpy as np
import pandas as pd
from scipy import stats

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.
EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong, tol):
            print("Hint:", message)
    return False


def check_all(label, known):
    """Compare several variables with their known answers: name -> (answer, tolerance). It never stops Run all."""
    got = {name: globals().get(name) for name in known}
    if all(value is None for value in got.values()):
        RESULTS[label] = None
        print(f"{label}: give " + ", ".join(known) + " a value, then run this cell again.")
        return False
    wrong = [name for name, (want, tol) in known.items() if got[name] is None or not same(got[name], want, tol)]
    RESULTS[label] = not wrong
    if wrong:
        print(f"{label}: not yet. Check " + ", ".join(wrong) + ".")
    else:
        print(f"{label}: correct.")
    return not wrong

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def read(name, **kwargs):
    try:
        return pd.read_csv(BASE + name, **kwargs)
    except Exception:
        return pd.read_csv(LOCAL + name, **kwargs)


users = read("users.csv", parse_dates=["signup_date"])
assign = read("experiment_assignments.csv", parse_dates=["assigned_at"])
subs = read("subscriptions.csv", parse_dates=["started_on", "ended_on", "updated_at"])
events = read("events.csv", parse_dates=["event_ts"], usecols=["user_id", "event_ts", "event_name", "platform"])    # 13 MB: a few seconds

# Known answer from the dataset README: 21,841 people, 15,868 experiment rows and 3,840 subscription rows.
assert (len(users), len(assign), len(subs)) == (21841, 15868, 3840)

# The platform of a person is the platform of their signup_complete event; its time is the signup time.
signup = events[events["event_name"] == "signup_complete"].drop_duplicates("user_id").set_index("user_id")
users["platform"] = users["user_id"].map(signup["platform"])
users["signup_ts"] = users["user_id"].map(signup["event_ts"])


def two_prop_p(c0, n0, c1, n1):
    """Two-sided p-value of the z test of two proportions with a pooled standard error (lesson C9); works on arrays too."""
    c0, n0, c1, n1 = (np.asarray(v, dtype=float) for v in (c0, n0, c1, n1))
    pooled = (c0 + c1) / (n0 + n1)
    se = np.sqrt(pooled * (1 - pooled) * (1 / n0 + 1 / n1))
    return 2 * stats.norm.sf(np.abs((c1 / n1 - c0 / n0) / se))


def gap_table(control, variant):
    """Rates, the gap, its 95 percent interval and the p-value for two 0/1 columns (lesson C9). Everything is a share, not points."""
    n0, n1 = len(control), len(variant)
    c0, c1 = int(control.sum()), int(variant.sum())
    p0, p1 = c0 / n0, c1 / n1
    se = np.sqrt(p0 * (1 - p0) / n0 + p1 * (1 - p1) / n1)
    return {"n0": n0, "n1": n1, "c0": c0, "c1": c1, "rate0": p0, "rate1": p1, "gap": p1 - p0, "low": p1 - p0 - 1.96 * se, "high": p1 - p0 + 1.96 * se, "p": float(two_prop_p(c0, n0, c1, n1))}


print(f"Data loaded: {len(users):,} people, {len(assign):,} experiment rows.")

## Unit 1: The plan is written before the data

**Predict first.** paywall_v2 was a test of a redesigned paywall. Of the people who saw the paywall at least once, what share saw it more than once: about 1 in 20, 1 in 5 or 1 in 2?

In [ ]:
print(int(assign.groupby("user_id").size().max()), assign["user_id"].nunique(), len(assign))      # most rows for one person, people, rows
lag = assign.merge(users[["user_id", "signup_ts"]], on="user_id", validate="many_to_one")
seconds = (lag["assigned_at"] - lag["signup_ts"]).dt.total_seconds()                                  # from signup to the coin toss
print(int(seconds.min()), int(seconds.max()))
pw = assign[assign["experiment"] == "paywall_v2"]
views = events[events["event_name"].isin(["view_paywall", "paywall_viewed"]) & events["user_id"].isin(pw["user_id"])]
views = views.drop_duplicates(["user_id", "event_ts"])                                                # a client retry wrote some events twice
per_person = views.groupby("user_id").size()                                                          # paywall views of each person who saw it
print(len(pw), len(per_person), int((per_person >= 2).sum()), round(float((per_person >= 2).mean()), 3))

## Unit 2: How many people, and for how long

**Predict first.** In lesson C9 the team's bar for paywall_v2 was a lift of 1 point, and the baseline is 12.4 percent. How many people per group would a test need to catch a true lift of exactly 1 point four times in five: about 5,000, 10,000 or 20,000? Then: paywall_v2 had 2,706 and 2,713 people. If the true lift had been exactly 2 points, in about how many of 100 such tests would p come out below 0.05?

In [ ]:
def people_per_group(p0, lift, alpha=0.05, power=0.80):
    """People needed in EACH group to catch a true lift of `lift` (0.02 means 2 points) from a baseline `p0`."""
    p_bar = p0 + lift / 2                                           # the average of the baseline and the baseline plus the lift
    z_alpha = stats.norm.ppf(1 - alpha / 2)                         # 1.96 for a two-sided bar of 0.05
    z_power = stats.norm.ppf(power)                                 # 0.84 for 80 percent power
    return int(np.ceil(2 * p_bar * (1 - p_bar) * (z_alpha + z_power) ** 2 / lift ** 2))


baseline = 0.124                                                    # an assumption: the share who buy Plus within 14 days, as in lesson C8
days_run = (pw["assigned_at"].max().normalize() - pw["assigned_at"].min().normalize()).days + 1
per_day = len(pw) / days_run                                        # people paywall_v2 enrolled each day
print(days_run, round(float(per_day), 1))
for lift in (0.01, 0.02, 0.03):
    n = people_per_group(baseline, lift)
    days = 2 * n / per_day
    print(f"lift +{100 * lift:.0f}: {n:,} per group, {2 * n:,} in all, {days:.0f} days, {int(np.ceil(days / 7))} whole weeks")
print(round(16 * baseline * (1 - baseline) / 0.02 ** 2))            # the rule of thumb 16 * variance / lift squared, for 2 points

In [ ]:
rng = np.random.default_rng(14)
runs = 100_000
n_control, n_variant = int((pw["variant"] == "control").sum()), int((pw["variant"] == "variant").sum())
for lift in (0.01, 0.02, 0.03):                                     # the true lift of each simulated world
    buyers0 = rng.binomial(n_control, baseline, size=runs)
    buyers1 = rng.binomial(n_variant, baseline + lift, size=runs)
    caught = two_prop_p(buyers0, n_control, buyers1, n_variant) < 0.05
    print(f"lift +{100 * lift:.0f}: {n_control:,} and {n_variant:,} people catch it in {caught.mean():.2f} of {runs:,} tests")
n_plan = people_per_group(baseline, 0.02)
buyers0 = rng.binomial(n_plan, baseline, size=runs)
buyers1 = rng.binomial(n_plan, baseline + 0.02, size=runs)
print(n_plan, round(float((two_prop_p(buyers0, n_plan, buyers1, n_plan) < 0.05).mean()), 2))

## Unit 3: Count the groups before you read anything

**Predict first.** The coin gives every new person control or variant with a chance of one half. The cell counts the people in each group of the three experiments and tests the counts against a fair 50/50 split with a chi-square test. For which of the three experiments will the split look wrong? What is the p-value there?

In [ ]:
for name, g in assign.groupby("experiment"):
    n_control = int((g["variant"] == "control").sum())
    n_variant = int((g["variant"] == "variant").sum())
    srm = stats.chisquare([n_control, n_variant])                   # observed counts against equal counts, the default
    print(name, n_control, n_variant, round(float(srm.statistic), 2), round(float(srm.pvalue), 4))

## Unit 4: Who is missing

**Predict first.** Take everyone who signed up while signup_flow_v2 ran (May 4 to June 7) and set aside the people who have a row in the experiment table. Who is left over: people of every platform or of one? How many events does each of them have?

In [ ]:
sf = assign[assign["experiment"] == "signup_flow_v2"]
window = users[(users["signup_date"] >= "2026-05-04") & (users["signup_date"] <= "2026-06-07")]      # everyone who signed up while the test ran
missing = window[~window["user_id"].isin(sf["user_id"])]
print(len(window), len(sf), len(missing))
print(missing["platform"].value_counts().to_dict())
print(events[events["user_id"].isin(missing["user_id"])].groupby("user_id").size().value_counts().to_dict())     # events of each missing person
android = sf.merge(users[["user_id", "platform"]], on="user_id", validate="one_to_one")
android = android[android["platform"] == "android"]
a_control = int((android["variant"] == "control").sum())
a_variant = int((android["variant"] == "variant").sum())
print(a_control, a_variant, a_variant + len(missing))                                                   # Android people in each group, and the variant with the missing added
naive = gap_table(sf.loc[sf["variant"] == "control", "converted"], sf.loc[sf["variant"] == "variant", "converted"])
print(f"{100 * naive['rate0']:.1f}% and {100 * naive['rate1']:.1f}%, gap {100 * naive['gap']:+.2f} points, p = {naive['p']:.3f}")

## Unit 5: Does the lift last

**Predict first.** Cut paywall_v2 by the week the person signed up: week 1 is January 12 to 18, weeks 2 to 8 are the rest. Will the gap in week 1 be bigger than, smaller than or about the same as the gap in the weeks after?

In [ ]:
pw = pw.copy()
pw["week"] = (pw["assigned_at"].dt.normalize() - pd.Timestamp("2026-01-12")).dt.days // 7 + 1      # week 1 is January 12 to 18
parts = {"week 1": pw[pw["week"] == 1], "weeks 2 to 8": pw[pw["week"] >= 2], "all eight weeks": pw}
tables = {}
for label, g in parts.items():
    t = gap_table(g.loc[g["variant"] == "control", "converted"], g.loc[g["variant"] == "variant", "converted"])
    tables[label] = t
    print(f"{label}: {t['n0']:,} and {t['n1']:,} people, {100 * t['rate0']:.1f}% and {100 * t['rate1']:.1f}%, gap {100 * t['gap']:+.1f} points ({100 * t['low']:.1f} to {100 * t['high']:.1f}), p = {t['p']:.3f}")

## Unit 6: The readout against the rule

**Predict first.** The rule written before launch (an assumption of this example): ship if the lift is at least 2 points and its interval stays above 0 and the interval of the cancel guardrail stays below +5 points; drop if even the top of the lift's interval is below 2 points; otherwise keep running. The guardrail is the share of buyers who canceled within 30 days of starting. Which verdict does paywall_v2 get: ship, drop or keep running?

In [ ]:
latest = subs.sort_values("updated_at").groupby("subscription_id").tail(1)                    # the current row of each subscription
first_sub = latest.sort_values("started_on").groupby("user_id").head(1)
buyers = pw[pw["converted"]].merge(first_sub[["user_id", "started_on", "ended_on"]], on="user_id", how="left", validate="one_to_one")
buyers["canceled_30"] = (buyers["ended_on"] - buyers["started_on"]).dt.days <= 30           # a running subscription has no end date: False
cancel = gap_table(buyers.loc[buyers["variant"] == "control", "canceled_30"], buyers.loc[buyers["variant"] == "variant", "canceled_30"])
print(cancel["n0"], cancel["n1"], cancel["c0"], cancel["c1"])
print(f"{100 * cancel['rate0']:.1f}% and {100 * cancel['rate1']:.1f}%, gap {100 * cancel['gap']:+.1f} points ({100 * cancel['low']:.1f} to {100 * cancel['high']:.1f}), p = {cancel['p']:.3f}")


def verdict(lift, lift_low, lift_high, cancel_high, bar=0.02, limit=0.05):
    """The rule written before launch: ship, drop or keep running."""
    if lift_high < bar:
        return "drop"                                               # even the best case is below the bar
    if lift >= bar and lift_low > 0 and cancel_high < limit:
        return "ship"
    return "keep running"


main = tables["all eight weeks"]
print(verdict(main["gap"], main["low"], main["high"], cancel["high"]))

## Finish it

### Exercise 1: Finish it

**Task:** A redesign is planned for people who sign up on Android only. Their 14-day conversion is 7.3 percent (the dataset README), and the smallest lift worth catching is 2 points. Finish the calculation: how many people does the test need in each group to catch a true lift of 2 points with 80 percent power at the bar 0.05? Round up to a whole person.

**Expected:** `answer_1` is a whole number of people per group.

In [ ]:
p0_android = 0.073                                                  # given: 14-day conversion of people who signed up on Android
lift_android = 0.02                                                 # given: the smallest lift worth catching, 2 points
z_alpha = stats.norm.ppf(0.975)                                     # given: 1.96
z_power = stats.norm.ppf(0.80)                                      # given: 0.84
p_bar = p0_android + lift_android / 2                               # given: the average of the baseline and the baseline plus the lift
# people needed in EACH group, rounded up to a whole person
answer_1 = None  # replace None with the number

In [ ]:
_ = check("Exercise 1", "answer_1", 2987, traps={5974: "That is the two groups together. The formula gives the people needed in each group.", 1: "Use the lift as a share (0.02), as in the starter, not as points."})

<details>
<summary>Hint</summary>

The cell of unit 2 had the formula inside `people_per_group`: `2 * p_bar * (1 - p_bar) * (z_alpha + z_power) ** 2 / lift ** 2`, rounded up with `np.ceil`. Here the pieces are already named.

</details>

<details>
<summary>Half-solution</summary>

```python
answer_1 = int(np.ceil(2 * p_bar * (1 - p_bar) * (z_alpha + z_power) ** 2 / ...))
```

</details>

<details>
<summary>Solution</summary>

```python
answer_1 = int(np.ceil(2 * p_bar * (1 - p_bar) * (z_alpha + z_power) ** 2 / lift_android ** 2))
```

At a baseline of 7.3 percent, a 2-point lift needs 2,987 people in each group, 5,974 in all. A lower baseline needs fewer people for the same lift in points, because the rates vary less.

</details>

## On your own

### Exercise 2: On your own

**Task:** The cancel guardrail of paywall_v2 (buyers who canceled within 30 days of starting) had a baseline of 14.0 percent, and the team's limit is a rise of 5 points. Before you start, say in one sentence what the result looks like: a number of buyers in each group and a number of whole weeks, both much larger than what paywall_v2 had. How many buyers does each group need to catch a true rise of 5 points with 80 percent power at the bar 0.05? And how many whole weeks would paywall_v2's traffic take to supply them, at the average number of buyers a day in one group over its 56 days (the buyers of the two groups, divided by 2 and by the days)? Write the code in the empty cell of the notebook. Set `answer_2` to the list [buyers per group, whole weeks].

**Expected:** `answer_2` is a list of two whole numbers.

In [ ]:
_ = check("Exercise 2", "answer_2", [866, 19], traps={(866, 10): "That counts the buyers of both groups together. Each group needs its own 866, so divide the buyers of the two groups by 2.", (433, 10): "The formula gives the people needed in each group. Use it as it is."})

<details>
<summary>Hint</summary>

Call `people_per_group(0.14, 0.05)` from unit 2: for this guardrail the "people" are buyers. Then divide by the buyers a day in one group, `pw["converted"].sum() / 2 / days_run`, and by 7, and round up.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"About eight hundred and some buyers in each group, and nearly twenty weeks, more than twice the eight weeks of the test."*

```python
needed = people_per_group(0.14, 0.05)
buyers_a_day = ...
```

</details>

<details>
<summary>Solution</summary>

```python
needed = people_per_group(0.14, 0.05)
buyers_a_day = int(pw["converted"].sum()) / 2 / days_run
answer_2 = [needed, int(np.ceil(needed / buyers_a_day / 7))]
```

Each group needs 866 buyers, and paywall_v2 had 343 and 415. At 6.8 buyers a day in a group that takes 128 days, 19 whole weeks. A guardrail on buyers is slow because only about one person in seven buys.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague checks the split of signup_flow_v2 among the people who came through organic search. The counts are given. The colleague puts the split in shares "so that it works for any size" and reports p = 0.93: a perfect split. Fix the cell so that `answer_3` is the p-value of the chi-square test of the counts, rounded to 4 decimals.

**Expected:** `answer_3` is a number between 0 and 1.

In [ ]:
sf_org = sf.merge(users[["user_id", "channel"]], on="user_id", validate="one_to_one")
sf_org = sf_org[sf_org["channel"] == "organic_search"]
n_c = int((sf_org["variant"] == "control").sum())                    # given: people in control
n_v = int((sf_org["variant"] == "variant").sum())                    # given: people in the variant
share = np.array([n_c, n_v]) / (n_c + n_v)                           # colleague: "put the split in shares so that it works for any size"
srm_p = float(stats.chisquare(share).pvalue)
answer_3 = round(srm_p, 4)

In [ ]:
_ = check("Exercise 3", "answer_3", 0.0027, traps={0.9296: "That is the test of the shares. The chi-square test needs the counts: the same split in shares is a sample of one person.", 0.0028: "Round the p-value to 4 decimals."}, tol=5e-5)

<details>
<summary>Hint</summary>

Print `share` and `n_c + n_v`. The statistic adds up (observed − expected)² / expected: what happens to it when the observed values are shares of 1 instead of counts of people?

</details>

<details>
<summary>Half-solution</summary>

```python
srm_p = float(stats.chisquare([n_c, ...]).pvalue)
```

</details>

<details>
<summary>Solution</summary>

```python
srm_p = float(stats.chisquare([n_c, n_v]).pvalue)
answer_3 = round(srm_p, 4)
```

Among organic search people, 628 are in control and 526 in the variant: p = 0.0027. The test of shares gave 0.93 because a split of 0.54 and 0.46 of one person is no evidence at all; the statistic grows with the number of people. 0.0027 is below 0.05 and above the alarm bar of 0.0005 used in the lesson: a hint to look at, not an alarm.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")